In [1]:
import pandas as pd 
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    roc_curve,
    roc_auc_score
)
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [2]:
Prestamos = pd.read_excel("Prestamos_Data_Alumnos_v5.xlsx")   
Diccionario_variables_alumnos = pd.read_excel("Diccionario_Variables_Alumnos.xlsx")

In [3]:
#filtrar el proposito por educacion
Prestamos_educacion = Prestamos[Prestamos["Proposito"] == "Educación"]

In [4]:
# pasamos las comumnas a tipo int porque son categoricas y no tienen decimales, por lo que no tiene sentido que sean float.
Prestamos_educacion['Posesion_Hipoteca'] = Prestamos_educacion['Posesion_Hipoteca'].astype(int)
Prestamos_educacion['Personas_Cargo'] = Prestamos_educacion['Personas_Cargo'].astype(int)
Prestamos_educacion['Fiador'] = Prestamos_educacion['Fiador'].astype(int)
Prestamos_educacion['Duracion'] = Prestamos_educacion['Duracion'].astype(int)
Prestamos_educacion['Meses_Empleo'] = Prestamos_educacion['Meses_Empleo'].astype(int)
Prestamos_educacion['Ingresos'] = Prestamos_educacion['Ingresos'].astype(int)

C:\Users\Oier\AppData\Local\Temp\ipykernel_27392\1568429427.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Prestamos_educacion['Posesion_Hipoteca'] = Prestamos_educacion['Posesion_Hipoteca'].astype(int)
C:\Users\Oier\AppData\Local\Temp\ipykernel_27392\1568429427.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Prestamos_educacion['Personas_Cargo'] = Prestamos_educacion['Personas_Cargo'].astype(int)
C:\Users\Oier\AppData\Local\Temp\ipykernel_27392\1568429427.py:4: SettingWithCopyWarning: 
A value is 

In [5]:
#esta columna es ruido no aporta nada al modelo, por lo que la eliminamos
Prestamos_educacion = Prestamos_educacion.drop(columns=["Proposito"])

In [6]:
Prestamos_educacion.duplicated().sum()

np.int64(1)

In [7]:
#eliminar los duplicados de la base de datos Prestamos_educacion por lo que no aporta nada al modelo, por lo que la eliminamos 
Prestamos_educacion.drop_duplicates(inplace=True)

In [8]:
#aunque haya pocos valores nulos,vamos a tratarlos en vez de eliminarlos, ya que no queremos perder datos, por lo que vamos a rellenarlos con la mediana de la columna.
Prestamos_educacion["Ratio_Interes"] = Prestamos_educacion["Ratio_Interes"].fillna(
    Prestamos_educacion["Ratio_Interes"].median())

In [9]:
Prestamos_educacion["Estudios"] = Prestamos_educacion["Estudios"].fillna(
    Prestamos_educacion["Estudios"].mode()[0])

In [10]:
orden_estudios = {
    "Escolar": 0,
    "Grado Universitario": 1,
    "Máster": 2,
    "Doctorado": 3
}

Prestamos_educacion["Estudios"] = Prestamos_educacion["Estudios"].str.strip()
Prestamos_educacion["Estudios_encoded"] = Prestamos_educacion["Estudios"].map(orden_estudios)

print(Prestamos_educacion[["Estudios", "Estudios_encoded"]].value_counts())

Estudios             Estudios_encoded
Escolar              0                   69818
Grado Universitario  1                   66497
Máster               2                   28091
Doctorado            3                   14202
Name: count, dtype: int64


In [11]:
Prestamos_educacion = Prestamos_educacion.drop(columns=["Estudios"])

In [12]:
Prestamos_educacion["Estado_Civil"] = Prestamos_educacion["Estado_Civil"].str.strip().str.capitalize()
Prestamos_educacion = pd.get_dummies(
    Prestamos_educacion,
    columns=["Estado_Civil"],
    dtype=int,
    drop_first=True
)

In [13]:
Prestamos_educacion["Tipo_Jornada_Laboral"] = Prestamos_educacion["Tipo_Jornada_Laboral"].str.strip().str.capitalize()
Prestamos_educacion = pd.get_dummies(
    Prestamos_educacion,
    columns=["Tipo_Jornada_Laboral"],
    dtype=int,
    drop_first=True
)

In [ ]:
porcentaje = pd.DataFrame({
        "Frecuencia": Prestamos_educacion['Impago'].value_counts(),
        "Porcentaje": (
            Prestamos_educacion['Impago']
            .value_counts(normalize=True)
            .mul(100)
            .round(2)
        )
    })
porcentaje

,Frecuencia,Porcentaje
Impago,,
0,157833,88.37
1,20775,11.63


La variable impago

## Modelo

el modelo usado

In [15]:
copia= Prestamos_educacion.copy()
#hacemos una copia de los datos para poder probar diferentes modelos y no perder los datos originales.

In [17]:
X= copia.drop(columns=["Impago","ID","Prima"])
y= copia["Impago"]